# Model Selection — 250 points, small network (64→16→10), 300 seeds

Within-regime test. Theory says unseen-only L2 regions = weakness.
Result: k_free rho=+0.117, p=0.043.

**Setup:** A100, High RAM, Run all.

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import numpy as np, time, json, copy
from torchvision import datasets, transforms
from scipy.stats import spearmanr
from collections import Counter

DEVICE = torch.device('cuda')
props = torch.cuda.get_device_properties(0)
mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
print(f'GPU: {torch.cuda.get_device_name(0)}, {mem/1e9:.0f} GB')
torch.manual_seed(42); np.random.seed(42)

transform = transforms.Compose([transforms.ToTensor(),
                                 transforms.Normalize((0.1307,),(0.3081,))])
train_ds = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_ds = datasets.MNIST('./data', train=False, download=True, transform=transform)

X_train_full = train_ds.data.float().view(-1,784).to(DEVICE)/255.0
y_train_full = train_ds.targets.to(DEVICE)
X_test = test_ds.data.float().view(-1,784).to(DEVICE)/255.0
y_test = test_ds.targets.to(DEVICE)

N_CHILD = 250
child_idx = torch.randperm(len(X_train_full),
                           generator=torch.Generator().manual_seed(42))[:N_CHILD]
X_train = X_train_full[child_idx]
y_train = y_train_full[child_idx]

X_pool = torch.cat([X_train_full, X_test])
y_pool = torch.cat([y_train_full, y_test])
unseen_mask = torch.ones(len(X_pool), dtype=torch.bool, device=DEVICE)
unseen_mask[child_idx] = False
X_unseen = X_pool[unseen_mask]
y_unseen = y_pool[unseen_mask]

print(f'Child: {len(X_train)}, Unseen: {len(X_unseen)}, Test: {len(X_test)}')

In [ ]:
D1, D2 = 64, 16
N_NETS = 300
BATCH_SIZE = 64
LR = 0.1
MAX_EPOCHS = 500

class MLP3(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, D1)
        self.fc2 = nn.Linear(D1, D2)
        self.fc3 = nn.Linear(D2, 10)
    def forward(self, x):
        return self.fc3(torch.relu(self.fc2(torch.relu(self.fc1(x)))))

def train_model(seed):
    torch.manual_seed(seed)
    model = MLP3().to(DEVICE)
    opt = optim.SGD(model.parameters(), lr=LR)
    loss_fn = nn.CrossEntropyLoss()
    N = len(X_train)
    for ep in range(MAX_EPOCHS):
        perm = torch.randperm(N, device=DEVICE)
        for s in range(0, N, BATCH_SIZE):
            idx = perm[s:s+BATCH_SIZE]
            loss = loss_fn(model(X_train[idx]), y_train[idx])
            opt.zero_grad(); loss.backward(); opt.step()
        if (ep+1) % 10 == 0:
            with torch.no_grad():
                if (model(X_train).argmax(1)==y_train).float().mean().item() >= 0.9999:
                    break
    return model

@torch.no_grad()
def accuracy(model, X, y):
    model.eval(); a = (model(X).argmax(1)==y).sum().item()/len(X); model.train()
    return a

def hessian_trace(model, n_samples=30):
    loss_fn = nn.CrossEntropyLoss(); model.eval()
    traces = []
    for _ in range(n_samples):
        model.zero_grad()
        grads = torch.autograd.grad(loss_fn(model(X_train), y_train),
                                     model.parameters(), create_graph=True)
        vs = [torch.randint(0,2,p.shape,device=DEVICE).float()*2-1
              for p in model.parameters()]
        gv = sum((g*v).sum() for g,v in zip(grads,vs))
        hvps = torch.autograd.grad(gv, model.parameters())
        traces.append(sum((v*hv).sum().item() for v,hv in zip(vs,hvps)))
    model.train()
    return np.mean(traces)

@torch.no_grad()
def L2_pattern_count(model, X):
    model.eval()
    h1 = torch.relu(model.fc1(X))
    h2_pre = model.fc2(h1)
    pat = (h2_pre > 0).cpu().numpy()
    row_size = pat.shape[1]
    n = len(set(pat.tobytes()[i*row_size:(i+1)*row_size] for i in range(len(pat))))
    model.train()
    return n

@torch.no_grad()
def weakness_proxy(model):
    """Free parameters in underdetermined L2 regions on training data."""
    model.eval()
    h1 = torch.relu(model.fc1(X_train))
    h2_pre = model.fc2(h1)
    pat = (h2_pre > 0).cpu().numpy()
    
    row_size = pat.shape[1]
    pat_ids = [pat[j].tobytes() for j in range(len(pat))]
    counts = Counter(pat_ids)
    
    n_regions = len(counts)
    params_per_region = D2 * 10 + 10  # 170 for 16->10 affine
    
    # Free params: each region with k points has max(0, D2*10+10 - k) free
    total_free = sum(max(0, params_per_region - k) for k in counts.values())
    
    # Empty regions: regions with 0 training points
    # (need to count on unseen data too)
    h1_u = torch.relu(model.fc1(X_unseen))
    h2_u = model.fc2(h1_u)
    pat_u = (h2_u > 0).cpu().numpy()
    pat_ids_u = set(pat_u.tobytes()[i*row_size:(i+1)*row_size]
                    for i in range(len(pat_u)))
    pat_ids_t = set(pat_ids)
    
    # Unseen-only regions: regions that have unseen points but no training points
    unseen_only = pat_ids_u - pat_ids_t
    n_unseen_only = len(unseen_only)
    
    model.train()
    return n_regions, total_free, n_unseen_only

def apply_T_beta(model, beta):
    m = copy.deepcopy(model)
    with torch.no_grad():
        m.fc1.weight.mul_(beta); m.fc1.bias.mul_(beta)
        m.fc2.weight.mul_(1.0/beta)
    return m

print('Ready.')

In [ ]:
######################################################################
# TRAIN 300 NETWORKS
######################################################################
networks = []
print(f'Training {N_NETS} networks (bs={BATCH_SIZE}, lr={LR})...')
t0_all = time.time()

for seed in range(N_NETS):
    t0 = time.time()
    model = train_model(seed)
    tr = accuracy(model, X_train, y_train)
    te = accuracy(model, X_test, y_test)
    dt = time.time() - t0
    
    if True:
        print(f'  [{seed+1:3d}/{N_NETS}] train={tr:.4f} test={te:.4f} ({dt:.0f}s)')
    
    networks.append(dict(model=model, seed=seed,
                         train_acc=tr, test_acc=te))

ta_all = [n['test_acc'] for n in networks]
n_ok = sum(1 for n in networks if n['train_acc'] > 0.9999)
print(f'\nDone. {(time.time()-t0_all)/60:.1f} min.')
print(f'Memorised: {n_ok}/{N_NETS}')
print(f'Test acc: {np.mean(ta_all):.4f} ± {np.std(ta_all):.4f} '
      f'(range {np.min(ta_all):.4f} to {np.max(ta_all):.4f})')

In [ ]:
######################################################################
# COMPUTE ALL MEASURES
######################################################################
print('Computing measures...')
t0 = time.time()

for i, net in enumerate(networks):
    model = net['model']
    
    net['hessian'] = hessian_trace(model, n_samples=30)
    net['L2_unseen'] = L2_pattern_count(model, X_unseen)
    net['L2_test'] = L2_pattern_count(model, X_test)
    
    n_reg, total_free, n_unseen_only = weakness_proxy(model)
    net['L2_train_regions'] = n_reg
    net['free_params'] = total_free
    net['unseen_only_regions'] = n_unseen_only
    
    with torch.no_grad():
        net['weight_norm'] = sum(p.norm().item()**2
                                 for p in model.parameters())**0.5
        # L1 norm (sparsity proxy)
        net['weight_l1'] = sum(p.abs().sum().item()
                               for p in model.parameters())
    
    if (i+1) % 20 == 0:
        print(f'  [{i+1:3d}/{N_NETS}] test={net["test_acc"]:.4f} '
              f'hess={net["hessian"]:8.2f} L2u={net["L2_unseen"]:6d} '
              f'free={net["free_params"]:8d} unseen_only={net["unseen_only_regions"]:6d}')

print(f'Done. ({(time.time()-t0)/60:.1f} min)')

In [ ]:
######################################################################
# REPARAMETERISATION SANITY CHECK
######################################################################
print('Reparam check (nets 0, 50, 99, beta=10):')
for i in [0, 50, 99]:
    net = networks[i]
    mb = apply_T_beta(net['model'], 10.0)
    
    te_b = accuracy(mb, X_test, y_test)
    ht_b = hessian_trace(mb, n_samples=20)
    l2_b = L2_pattern_count(mb, X_unseen)
    _, free_b, uo_b = weakness_proxy(mb)
    
    print(f'  Net {i}: test {net["test_acc"]:.4f}->{te_b:.4f}  '
          f'hess {net["hessian"]:.1f}->{ht_b:.1f} ({ht_b/net["hessian"]:.0f}x)  '
          f'L2 {net["L2_unseen"]}->{l2_b}  '
          f'free {net["free_params"]}->{free_b}  '
          f'uo {net["unseen_only_regions"]}->{uo_b}')
    del mb; torch.cuda.empty_cache()

In [ ]:
######################################################################
# THE RESULT
######################################################################
ta = np.array([n['test_acc'] for n in networks])
ht = np.array([n['hessian'] for n in networks])
l2u = np.array([n['L2_unseen'] for n in networks])
l2t = np.array([n['L2_train_regions'] for n in networks])
fp = np.array([n['free_params'] for n in networks])
uo = np.array([n['unseen_only_regions'] for n in networks])
wn = np.array([n['weight_norm'] for n in networks])
wl1 = np.array([n['weight_l1'] for n in networks])

print('='*65)
print('  MODEL SELECTION: Which measure picks the best network?')
print('='*65)
print(f'\n  300 networks, same config, different seeds.')
print(f'  Test acc range: {ta.min():.4f} to {ta.max():.4f} '
      f'({(ta.max()-ta.min())*10000:.0f} images of difference)')

measures = [
    ('Hessian trace (sharpness)', ht, 'NO'),
    ('Weight L2 norm (simplicity)', wn, 'NO'),
    ('Weight L1 norm (sparsity)', wl1, 'NO'),
    ('L2 unseen patterns', l2u, 'YES'),
    ('L2 train regions', l2t, 'YES'),
    ('Free parameters (weakness)', fp, 'YES'),
    ('Unseen-only regions', uo, 'YES'),
]

print(f'\n  {"Measure":>35s} {"rho":>8s} {"p-value":>12s} {"invariant":>10s}')
print('  ' + '-'*68)
for name, vals, inv in measures:
    if len(set(vals)) > 1:
        r, p = spearmanr(ta, vals)
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
        print(f'  {name:>35s} {r:+8.4f} {p:12.4e} {inv:>10s} {sig}')
    else:
        print(f'  {name:>35s} {"constant":>8s}')

print(f'\n  Significance: *** p<0.001, ** p<0.01, * p<0.05')

In [ ]:
######################################################################
# FIGURES
######################################################################
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

def plot_one(ax, x, y, xlabel, title, inv):
    ax.scatter(x, y, s=20, alpha=0.6, c='#006D5B' if inv else '#CC5500',
               edgecolors='k', lw=0.3)
    if len(set(x)) > 1:
        r, p = spearmanr(y, x)
        ax.text(0.05, 0.05, f'ρ={r:.3f}\np={p:.2e}',
                transform=ax.transAxes, fontsize=9,
                bbox=dict(boxstyle='round', fc='white', alpha=0.8))
    ax.set(xlabel=xlabel, ylabel='Test accuracy', title=title)
    ax.grid(True, alpha=0.3)

plot_one(axes[0,0], ht, ta, 'Hessian trace',
         'Sharpness (NOT invariant)', False)
plot_one(axes[0,1], wn, ta, 'Weight L2 norm',
         'Simplicity (NOT invariant)', False)
plot_one(axes[0,2], wl1, ta, 'Weight L1 norm',
         'Sparsity (NOT invariant)', False)
plot_one(axes[1,0], l2u, ta, 'L2 unseen patterns',
         'L2 diversity (invariant)', True)
plot_one(axes[1,1], fp, ta, 'Free parameters',
         'Weakness proxy (invariant)', True)
plot_one(axes[1,2], uo, ta, 'Unseen-only regions',
         'Unseen-only (invariant)', True)

plt.suptitle('Model selection: same config, different seeds (n=100)',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('fig_modelselect.pdf', bbox_inches='tight', dpi=200)
plt.savefig('fig_modelselect.png', bbox_inches='tight', dpi=200)
plt.show()
print('Saved fig_modelselect')

In [ ]:
######################################################################
# SAVE AND DOWNLOAD
######################################################################
class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.floating,)): return float(obj)
        if isinstance(obj, (np.integer,)): return int(obj)
        return super().default(obj)

save_nets = [{k:v for k,v in n.items() if k!='model'} for n in networks]
with open('modelselect_results.json', 'w') as f:
    json.dump(save_nets, f, indent=2, cls=NpEncoder)

from google.colab import files
for f in ['modelselect_results.json',
          'fig_modelselect.pdf', 'fig_modelselect.png']:
    try: files.download(f); print(f'Downloaded {f}')
    except Exception as e: print(f'Skip {f}: {e}')